# 판정기 학습 (Colab T4)

준비: `factcheck-judge-colab.zip` 을 내 드라이브 최상단(`MyDrive/`)에 올린다(같은 이름이 있으면 덮어쓰기). 런타임 → 런타임 유형 변경 → **T4 GPU**.

위에서부터 차례로 실행. 체크포인트·결과는 `MyDrive/factcheck-judge-runs/` 에 저장되니 연결이 끊겨도 **1·2번 셀 다시 실행 후 3번 셀의 `--resume`** 으로 이어서 한다.

T4 는 bf16 이 없어 `--fp16`(가중치 float32 + fp16 혼합정밀).

In [ ]:
# 1. 드라이브 연결 + 코드·데이터 풀기 + 패키지
NAME = 'q06b-v5'  # 이번 학습 이름 — 결과 폴더·zip 이름에 쓰인다
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/fj && mkdir -p /content/fj /content/drive/MyDrive/factcheck-judge-runs
!unzip -q /content/drive/MyDrive/factcheck-judge-colab.zip -d /content/fj
!ln -sfn /content/drive/MyDrive/factcheck-judge-runs /content/fj/runs
!pip -q install 'transformers>=5.0' 'peft>=0.13' 'accelerate>=1.0'
!pip -q uninstall -y torchao  # Colab 기본 torchao(0.10)가 최신 peft 와 충돌 — 우리 학습엔 필요 없음
!nvidia-smi --query-gpu=name,memory.total --format=csv
!cat /content/fj/DATA_VERSION.txt

In [ ]:
# 2. 작업 폴더
%cd /content/fj/scripts

In [ ]:
# 3. 학습 (끊긴 뒤 이어서 할 때는 끝에 --resume 추가)
!python train.py --base Qwen/Qwen3-0.6B --name {NAME} --fp16

In [ ]:
# 4. 평가 — 주장 단위(29·18건) → eval.json, 틀린 용어 60쌍 → eval_termswap.json (둘 다 runs/NAME/ 에 저장)
!python evaluate.py --base Qwen/Qwen3-0.6B --adapter ../runs/{NAME}/adapter
!python eval_termswap.py --base Qwen/Qwen3-0.6B --adapter ../runs/{NAME}/adapter

In [ ]:
# 5. 어댑터 압축 → MyDrive/factcheck-judge-runs/NAME-adapter.zip — 드라이브에서 '링크가 있는 모든 사용자'로 공유해 링크를 Claude 에게
!cd ../runs && rm -f {NAME}-adapter.zip && zip -qr {NAME}-adapter.zip {NAME}/adapter {NAME}/metrics.json {NAME}/eval.json {NAME}/eval_termswap.json && ls -l {NAME}-adapter.zip